# UCS × BFS: rotas em transporte público com custos diferentes

**Partes 4 e 5 da apresentação**: implementação dos dois algoritmos e análise comparativa dos resultados.

> ⚠️ **Aviso:** os bairros e pontos de referência são de **João Pessoa (PB)**, mas **as linhas, os tempos de viagem e as tarifas são fictícios**. Os valores foram escolhidos para ilustrar as diferenças entre os algoritmos e não representam a rede real de transporte da cidade.

**Roteiro**

1. Introdução
2. Modelagem: grafo, funções de custo e desenho
3. **Parte 4: implementação**: BFS, UCS, demonstração e verificação de corretude
4. **Parte 5: análise e comparação**: várias viagens, custo computacional, experimento de escala, quadro teórico e conclusões

## 1. Introdução

Planejar uma viagem de ônibus, trem ou a pé é um problema de **busca de caminho em grafo**: cada parada é um nó e cada trecho que dá para fazer num único embarque é uma aresta.

Os dois algoritmos comparados aqui respondem a perguntas diferentes:

| Algoritmo | O que minimiza | Como trata os custos |
|---|---|---|
| **BFS** (busca em largura) | o **número de trechos** (arestas) | **ignora** tempo, tarifa e qualquer outro peso |
| **UCS** (busca de custo uniforme) | o **custo acumulado** $g(n)$ da origem até o nó | usa a função de custo que recebermos: tempo, tarifa ou uma combinação |

Quando todos os trechos "custam" o mesmo, as duas respostas coincidem. Em transporte público, isso quase nunca acontece: um ônibus direto pode levar 40 minutos, enquanto uma caminhada seguida de dois ônibus curtos leva 32. A pergunta deste notebook é: **quanto se perde ao usar a BFS nesse domínio, e quanto custa, computacionalmente, fazer a escolha certa com a UCS?**

In [ ]:
# --- Estruturas de dados usadas pelos algoritmos (implementados do zero) ---
from collections import deque        # fila FIFO com popleft em O(1): é a fronteira da BFS
import heapq                         # heap binário (fila de prioridade): é a fronteira da UCS
import itertools                     # itertools.count() gera o contador de desempate da heap
from dataclasses import dataclass    # para agrupar o resultado de uma busca num objeto legível
import math                          # math.inf como "custo ainda desconhecido" e math.isclose p/ comparar floats

# --- Experimentos e medições ---
import random                        # grades aleatórias do experimento de escala (com semente fixa)
import time                          # perf_counter_ns: relógio de alta resolução para cronometrar

# --- Análise e visualização (fora dos algoritmos) ---
import pandas as pd                  # tabelas de resultados
import matplotlib.pyplot as plt      # gráficos
from matplotlib.lines import Line2D  # itens de legenda "manuais" para o desenho do grafo
from matplotlib.patches import Patch # itens de legenda com hachura para o gráfico de barras
import networkx as nx                # SÓ para desenhar o grafo e para validar a UCS com Dijkstra

# Configurações de exibição: tabelas mais largas e figuras com resolução boa para o GitHub
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({
    "figure.dpi": 110,               # nitidez razoável sem deixar o .ipynb pesado demais
    "axes.spines.top": False,        # eixos "limpos": só esquerda e base
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,               # grade discreta, para não competir com os dados
    "font.size": 10,
})

# Paleta categórica validada para daltonismo (azul, laranja, verde-água).
# A MESMA cor representa o MESMO algoritmo em todas as figuras do notebook.
COR_BFS = "#2a78d6"
COR_UCS_TEMPO = "#eb6834"
COR_UCS_TARIFA = "#1baf7a"

## 2. Modelagem

### 2.1 Dados da rede

O grafo é **não-direcionado**: dá para ir e voltar por qualquer trecho com o mesmo tempo e a mesma tarifa. Cada aresta representa **um embarque sem baldeação**, ou seja, um trecho de uma linha de ônibus, do trem ou uma caminhada.

Cada aresta tem quatro atributos:

- `tempo`: minutos de viagem;
- `tarifa`: reais pagos pelo embarque (caminhada = R$ 0,00; trem = R$ 0,50; ônibus = R$ 5,00);
- `modo`: `ônibus`, `trem` ou `a pé`;
- `linha`: identificador (fictício) da linha.

In [ ]:
# Cada tupla: (parada A, parada B, tempo em min, tarifa em R$, modo, linha).
# A ORDEM importa: ela define a ordem dos vizinhos na lista de adjacência e, portanto,
# a ordem em que a BFS descobre os nós (e qual rota ela devolve quando há empate em nº de trechos).
ARESTAS = [
    ("UFPB", "Castelo Branco", 5, 0.00, "a pé", "caminhada"),
    ("UFPB", "Lagoa", 25, 5.00, "ônibus", "L101"),
    ("UFPB", "Bancários", 12, 5.00, "ônibus", "L201"),
    ("Castelo Branco", "Torre", 15, 5.00, "ônibus", "L102"),
    ("Castelo Branco", "Tambaú", 40, 5.00, "ônibus", "L510"),
    ("Torre", "Tambaú", 12, 5.00, "ônibus", "L103"),
    ("Torre", "Lagoa", 10, 5.00, "ônibus", "L102"),
    ("Bancários", "Mangabeira", 10, 5.00, "ônibus", "L201"),
    ("Bancários", "Cabo Branco", 20, 5.00, "ônibus", "L202"),
    ("Mangabeira", "Valentina", 12, 5.00, "ônibus", "L301"),
    ("Valentina", "Lagoa", 45, 5.00, "ônibus", "L302"),
    ("Cabo Branco", "Tambaú", 15, 0.00, "a pé", "caminhada"),
    ("Tambaú", "Manaíra", 8, 5.00, "ônibus", "L104"),
    ("Manaíra", "Bessa", 10, 5.00, "ônibus", "L104"),
    ("Bessa", "Cabedelo", 20, 5.00, "ônibus", "L105"),
    ("Lagoa", "Tambaú", 25, 5.00, "ônibus", "L106"),
    ("Lagoa", "Manaíra", 30, 5.00, "ônibus", "L107"),
    ("Lagoa", "Rodoviária", 8, 0.00, "a pé", "caminhada"),
    ("Lagoa", "Cruz das Armas", 15, 5.00, "ônibus", "L401"),
    ("Rodoviária", "Estação Ferroviária", 5, 0.00, "a pé", "caminhada"),
    ("Estação Ferroviária", "Bayeux", 12, 0.50, "trem", "Trem"),
    ("Estação Ferroviária", "Cabedelo", 45, 0.50, "trem", "Trem"),
    ("Cruz das Armas", "Bayeux", 15, 5.00, "ônibus", "L402"),
]

# Visão tabular dos dados, útil para conferir com os slides
pd.DataFrame(ARESTAS, columns=["parada A", "parada B", "tempo (min)", "tarifa (R$)", "modo", "linha"])

### 2.2 Lista de adjacência

Os algoritmos trabalham sobre uma **lista de adjacência** feita à mão, sem networkx:

```
parada -> [(vizinho, {"tempo": ..., "tarifa": ..., "modo": ..., "linha": ...}), ...]
```

É a representação natural para busca: para expandir um nó, basta percorrer a lista dos seus vizinhos, em tempo proporcional ao grau do nó.

In [ ]:
def construir_grafo(arestas):
    # dict comum (não defaultdict) para que consultar uma parada inexistente dê erro,
    # em vez de criar silenciosamente uma parada vazia
    grafo = {}
    for a, b, tempo, tarifa, modo, linha in arestas:
        # Os atributos ficam num dict para que as funções de custo possam escolher
        # qual campo usar, sem os algoritmos precisarem saber quais campos existem
        atributos = {"tempo": tempo, "tarifa": tarifa, "modo": modo, "linha": linha}
        # setdefault cria a lista de vizinhos na primeira vez que a parada aparece
        grafo.setdefault(a, []).append((b, atributos))
        # Grafo NÃO-direcionado: o mesmo trecho vale nos dois sentidos.
        # Os dois sentidos compartilham o MESMO dict de atributos, o que não é problema
        # porque os algoritmos só leem os atributos, nunca os alteram.
        grafo.setdefault(b, []).append((a, atributos))
    return grafo


grafo = construir_grafo(ARESTAS)

print(f"Paradas: {len(grafo)} | trechos (arestas não-direcionadas): {len(ARESTAS)}")
print(f"Grau médio: {sum(len(v) for v in grafo.values()) / len(grafo):.2f} vizinhos por parada\n")
# Mostra um pedaço da lista de adjacência para ilustrar o formato
for vizinho, atributos in grafo["UFPB"]:
    print(f"UFPB -> {vizinho:<15} {atributos}")

### 2.3 Funções de custo

A UCS recebe **a função de custo como parâmetro**. Assim, o mesmo algoritmo responde a perguntas diferentes:

| Função | Custo de um trecho | Pergunta que responde |
|---|---|---|
| `custo_tempo` | minutos | *Qual é a rota mais rápida?* |
| `custo_tarifa` | reais | *Qual é a rota mais barata?* |
| `custo_generalizado` | tempo + `VALOR_TEMPO` × tarifa | *Qual rota equilibra tempo e dinheiro?* |
| `custo_unitario` | 1 | *Qual rota tem menos trechos?* (a pergunta da BFS) |

O **custo generalizado** é comum em planejamento de transporte: ele converte dinheiro em tempo equivalente. Com `VALOR_TEMPO = 6`, o passageiro considera que pagar R\$ 1,00 "dói" tanto quanto 6 minutos a mais de viagem. Uma passagem de ônibus de R\$ 5,00 pesa, então, como 30 minutos.

In [ ]:
# Quantos minutos equivalem a R$ 1,00 para o passageiro (parâmetro de preferência, não um dado da rede)
VALOR_TEMPO = 6


def custo_tempo(aresta):
    # Minimizar a soma dos tempos = rota mais rápida
    return aresta["tempo"]


def custo_tarifa(aresta):
    # Minimizar a soma das tarifas = rota mais barata (caminhadas são grátis)
    return aresta["tarifa"]


def custo_generalizado(aresta):
    # Converte a tarifa em "minutos equivalentes" para somar grandezas na mesma unidade
    return aresta["tempo"] + VALOR_TEMPO * aresta["tarifa"]


def custo_unitario(aresta):
    # Todo trecho vale 1: a UCS passa a minimizar o nº de trechos, exatamente como a BFS
    return 1


# Os três critérios "reais" do domínio, usados nas comparações da Parte 5
CRITERIOS = {"tempo": custo_tempo, "tarifa": custo_tarifa, "generalizado": custo_generalizado}

# Exemplo: o mesmo trecho visto por cada função de custo
exemplo = grafo["UFPB"][1][1]   # UFPB – Lagoa (ônibus L101)
print("Trecho UFPB – Lagoa:", exemplo)
for nome, f in {**CRITERIOS, "unitario": custo_unitario}.items():
    print(f"  custo_{nome:<13} = {f(exemplo)}")

### 2.4 Desenho do grafo

O desenho usa o `networkx` apenas como ferramenta gráfica. As **posições são fixas** e aproximam a geografia de João Pessoa: o litoral (Cabo Branco, Tambaú, Manaíra, Bessa) fica a **leste**, Bayeux a **oeste** e Cabedelo ao **norte**. As distâncias no desenho **não** são proporcionais aos tempos.

- **Cor por modo**: ônibus em cinza-escuro, trem em violeta e caminhada em verde **pontilhado**.
- Cada aresta mostra o **tempo** e a **tarifa**.
- `desenhar_grafo(rotas=...)` destaca rotas encontradas. As rotas ficam por cima da rede: a primeira **larga e translúcida**, a segunda **mais fina, por cima dela**, para os trechos em comum continuarem visíveis.

In [ ]:
# Posições (x = oeste→leste, y = sul→norte) que aproximam o mapa da cidade
POSICOES = {
    "Cabedelo": (7.8, 10.2),
    "Bessa": (8.9, 8.4),
    "Manaíra": (9.4, 6.8),
    "Tambaú": (9.6, 5.0),
    "Cabo Branco": (10.1, 3.0),
    "Torre": (5.6, 4.1),
    "Castelo Branco": (7.6, 3.2),
    "UFPB": (6.6, 2.1),
    "Bancários": (7.9, 0.9),
    "Mangabeira": (8.9, -0.5),
    "Valentina": (5.3, -1.0),
    "Lagoa": (3.5, 5.0),
    "Rodoviária": (2.7, 6.2),
    "Estação Ferroviária": (2.0, 7.4),
    "Cruz das Armas": (2.3, 3.0),
    "Bayeux": (0.0, 5.0),
}
# Garante que nenhuma parada ficou sem posição (erro de digitação nos nomes, por exemplo)
assert set(POSICOES) == set(grafo), set(POSICOES) ^ set(grafo)

# Estilo de cada modo: cor + tipo de linha (a caminhada é pontilhada, então não depende só da cor)
ESTILO_MODO = {
    "ônibus": {"cor": "#52514e", "linha": "solid"},
    "trem": {"cor": "#4a3aa7", "linha": "dashdot"},
    "a pé": {"cor": "#008300", "linha": "dotted"},
}

# Grafo do networkx com os MESMOS dados. Serve para desenhar aqui e para validar a UCS na Parte 4.
G = nx.Graph()
for a, b, tempo, tarifa, modo, linha in ARESTAS:
    G.add_edge(a, b, tempo=tempo, tarifa=tarifa, modo=modo, linha=linha)


def brl(valor):
    # Formata em reais no padrão brasileiro (vírgula decimal): 5.0 -> "R$ 5,00"
    return f"R$ {valor:.2f}".replace(".", ",")


def desenhar_grafo(rotas=(), titulo="Rede de transporte (dados fictícios)", ax=None):
    # rotas: sequência de (Resultado, cor, rótulo). A 1ª é larga e translúcida; as seguintes, finas por cima.
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 10))

    # 1) Rede completa, um modo de cada vez (cada modo tem cor e tipo de linha próprios)
    for modo, estilo in ESTILO_MODO.items():
        arestas_modo = [(a, b) for a, b, d in G.edges(data=True) if d["modo"] == modo]
        nx.draw_networkx_edges(G, POSICOES, edgelist=arestas_modo, ax=ax, edge_color=estilo["cor"],
                               style=estilo["linha"], width=1.6)

    # 2) Destaque das rotas POR CIMA da rede (mas por baixo dos rótulos, desenhados depois)
    larguras = [13, 4.5, 2]               # cada rota seguinte é mais fina que a anterior
    transparencias = [0.5, 1.0, 1.0]     # a primeira é translúcida: a rede e a rota de cima continuam visíveis
    itens_legenda_rotas = []
    for i, (res, cor, rotulo) in enumerate(rotas):
        trechos = list(zip(res.caminho, res.caminho[1:]))  # pares consecutivos = arestas da rota
        nx.draw_networkx_edges(G, POSICOES, edgelist=trechos, ax=ax, edge_color=cor,
                               width=larguras[i], alpha=transparencias[i])
        itens_legenda_rotas.append(Line2D([], [], color=cor, lw=larguras[i] * 0.6,
                                          alpha=transparencias[i], label=rotulo))

    # 3) Rótulo de tempo e tarifa no meio de cada aresta
    rotulos_arestas = {(a, b): f"{d['tempo']} min\n{brl(d['tarifa'])}" for a, b, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, POSICOES, edge_labels=rotulos_arestas, ax=ax, font_size=7,
                                 font_color="#52514e", rotate=False,
                                 bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.85})

    # 4) Paradas por cima de tudo, com o nome dentro de uma "etiqueta"
    nx.draw_networkx_labels(G, POSICOES, ax=ax, font_size=8.5, font_weight="bold",
                            bbox={"boxstyle": "round,pad=0.3", "fc": "#f0efec", "ec": "#52514e", "lw": 0.8})

    # Legenda: modos (sempre) + rotas destacadas (se houver)
    itens_modos = [Line2D([], [], color=e["cor"], ls=e["linha"], lw=1.8, label=m) for m, e in ESTILO_MODO.items()]
    ax.legend(handles=itens_modos + itens_legenda_rotas, loc="lower left", frameon=True, fontsize=9)

    # Rosa dos ventos mínima, para reforçar que as posições seguem a geografia
    ax.annotate("N", xy=(0.97, 0.97), xytext=(0.97, 0.88), xycoords="axes fraction",
                ha="center", fontweight="bold", arrowprops={"arrowstyle": "-|>", "color": "#52514e"})
    ax.set_title(titulo, fontsize=13)
    ax.grid(False)       # grade não faz sentido num desenho de grafo
    ax.axis("off")
    ax.margins(0.06)     # folga para as etiquetas das bordas não serem cortadas
    return ax


desenhar_grafo()
plt.show()

---
# Parte 4: Implementação

Os dois algoritmos compartilham a mesma "moldura":

- a **fronteira** guarda os nós descobertos que ainda não foram expandidos;
- **expandir** um nó significa retirá-lo da fronteira e gerar os seus vizinhos;
- o dicionário **`pai`** guarda, para cada nó alcançado, de onde viemos e por qual trecho. Com ele, reconstruímos a rota no final.

A diferença está **em como a fronteira é ordenada** (FIFO × prioridade por custo) e **em que momento testamos se chegamos ao destino** (na geração × na expansão).

### 4.1 Resultado de uma busca e reconstrução da rota

Para comparar os algoritmos de forma justa, os dois devolvem o mesmo objeto `Resultado`, com três métricas de esforço:

| Métrica | Definição usada aqui |
|---|---|
| `expandidos` | nós retirados da fronteira que tiveram os vizinhos gerados |
| `gerados` | entradas inseridas na fronteira, incluindo a origem. Na UCS, contam também as reinserções de um nó com custo melhor |
| `max_fronteira` | maior tamanho que a fronteira atingiu, um indicador de uso de memória |

In [ ]:
@dataclass
class Resultado:
    algoritmo: str        # nome exibido nas tabelas, ex.: "BFS" ou "UCS (tempo)"
    caminho: list         # paradas da origem ao destino; lista vazia se não houver rota
    arestas: list         # atributos de cada trecho percorrido (len(arestas) == len(caminho) - 1)
    expandidos: int       # esforço: nós expandidos
    gerados: int          # esforço: inserções na fronteira
    max_fronteira: int    # memória: pico de tamanho da fronteira


def reconstruir(pai, destino):
    # Se o destino nunca foi alcançado, não há rota: devolve listas vazias em vez de falhar
    if destino not in pai:
        return [], []
    caminho, arestas = [destino], []
    no = destino
    # Sobe pelos "ponteiros" pai até a origem, que é o único nó com pai None
    while pai[no] is not None:
        anterior, atributos = pai[no]
        arestas.append(atributos)    # guarda o trecho usado para chegar em `no`
        caminho.append(anterior)
        no = anterior
    # A subida produziu a rota de trás para frente (destino → origem); invertemos
    caminho.reverse()
    arestas.reverse()
    return caminho, arestas

### 4.2 BFS: busca em largura

- **Fronteira:** `collections.deque` usada como **fila FIFO**. `append` insere no fim e `popleft` remove do início, ambos em $O(1)$. Uma `list` com `pop(0)` custaria $O(n)$ por remoção.
- **Conjunto de alcançados:** o próprio dicionário `pai`. Um nó entra em `pai` no momento em que é **gerado** e nunca é gerado de novo. Por isso não precisamos de um `set` separado de visitados.
- **Teste de objetivo na geração:** assim que o destino aparece como vizinho, a busca termina, **sem esperar ele sair da fila**.

**Por que o teste na geração é válido aqui?** A BFS expande os nós em ordem não-decrescente de profundidade, ou seja, de número de trechos desde a origem. Suponha que o destino seja gerado pela primeira vez ao expandirmos um nó de profundidade $k$. Então:

1. todos os nós de profundidade $< k$ já foram expandidos antes, e nenhum deles tinha o destino como vizinho. Se tivesse, o destino teria sido gerado antes;
2. logo, o destino **não** está a $\le k$ trechos da origem, e a profundidade $k+1$ encontrada agora é a **mínima possível**.

Testar só na expansão devolveria a mesma rota, mas antes a busca geraria **o nível $d$ inteiro** (até $b^{d}$ nós a mais, com $b$ = fator de ramificação). O teste antecipado vale porque, para a BFS, **todo trecho custa 1**. Com custos diferentes, esse argumento deixa de valer, como mostra a UCS a seguir.

In [ ]:
def bfs(grafo, origem, destino):
    # Caso trivial: já estamos no destino; rota de 0 trechos, nada a expandir
    if origem == destino:
        return Resultado("BFS", [origem], [], expandidos=0, gerados=1, max_fronteira=1)

    fronteira = deque([origem])   # fila FIFO: quem entrou primeiro (mais raso) sai primeiro
    pai = {origem: None}          # nó -> (nó anterior, atributos do trecho); também é o conjunto de alcançados
    expandidos, gerados, max_fronteira = 0, 1, 1   # a origem já conta como gerada e está na fronteira

    while fronteira:
        no = fronteira.popleft()  # o nó mais antigo da fila = o de menor profundidade ainda não expandido
        expandidos += 1
        for vizinho, atributos in grafo[no]:
            # Já alcançado antes? Então ele já tem um caminho com nº de trechos <= ao atual; ignoramos
            if vizinho in pai:
                continue
            pai[vizinho] = (no, atributos)   # marca como alcançado NA GERAÇÃO e guarda por onde chegamos
            gerados += 1
            # Teste de objetivo NA GERAÇÃO: válido porque todos os trechos "custam" 1 (ver explicação acima)
            if vizinho == destino:
                caminho, arestas = reconstruir(pai, destino)
                return Resultado("BFS", caminho, arestas, expandidos, gerados, max_fronteira)
            fronteira.append(vizinho)        # entra no FIM da fila: será expandido depois de todo o nível atual
            max_fronteira = max(max_fronteira, len(fronteira))

    # Fila vazia sem achar o destino: não existe rota (grafo desconexo)
    return Resultado("BFS", [], [], expandidos, gerados, max_fronteira)


# Teste rápido: rota com o menor número de trechos entre UFPB e Tambaú
res = bfs(grafo, "UFPB", "Tambaú")
print(" → ".join(res.caminho), f"| {len(res.arestas)} trechos | expandidos={res.expandidos}, gerados={res.gerados}")